In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
# 改了 src/ 里的 .py，notebook 不用重启就能用上新代码
from signal_mysql import database as mysql
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, db, geo, privacy, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT, AREA_COLOR, SEG_COLOR  # 配色常量
viz.apply_theme()                           # 字体、配色、坐标轴样式：一行
con = mysql.connect(config.DB_SCHEMA)       # 路径从包里来：notebook 在哪个文件夹打开都找得到库
DAY = config.ANALYSIS_DAY                   # 分析日期（口径常量）
NB = "D12_多日流水线与门禁"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"    # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
from signal_mysql import database as mysql
from contextlib import closing
from signal_insight import __main__ as cli_main, dws, gate, ingest, pipeline
def cli(*args):
    """和在终端里敲 python -m signal_insight … 一样。"""
    return cli_main.main(list(args))

settings = config.load_settings()
# 今天要接着 D11 的生产库往下做。还没有生产库、或者是 D11 的代码建的（没有入库指纹），就重建。
rebuild = True
if mysql.schema_exists(config.WAREHOUSE_SCHEMA):
    with closing(mysql.connect(config.WAREHOUSE_SCHEMA)) as c_:
        tables = {t for (t,) in c_.execute("SELECT table_name AS name FROM information_schema.tables WHERE table_schema=DATABASE()")}
    rebuild = "ods_load_log" not in tables
if rebuild:
    c_ = mysql.connect(config.DB_SCHEMA)
    c_.execute(f"DROP DATABASE IF EXISTS {mysql.ident(config.WAREHOUSE_SCHEMA)}"); c_.commit(); c_.close()
    mysql.engine(mysql.settings_url(config.WAREHOUSE_SCHEMA)).dispose()
    print("用今天的代码重做 D11 的四步：建库、跑 10-03、入库 10-04、跑 10-04")
    cli("init")
    cli("run", "--date", DAY)
    f04 = next((f for f in [config.LANDING_DIR / "ods_signal_2018-10-04.parquet",
                            config.LANDING_DIR / "ods_signal_2018-10-04.db"] if f.exists()), None)
    if f04:
        cli("ingest", str(f04)); cli("run", "--date", "2018-10-04")
    else:
        print("缺 10-04 分区文件：跳过入库与 10-04；后面的格子要等文件到位")
else:
    cli("init")                                 # 生产库已有：只补登记表
wh = db.connect(config.WAREHOUSE_SCHEMA)
ingest.registry(wh)[["dt", "is_synthetic", "rows_ods", "gate_status", "pipeline_status"]]

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
# ── 出图 D12_01：状态机图 ───────────────────────────────────────────
# 为什么画状态机：流水线的核心不是“跑一遍”，而是“任何时刻每个分区处在什么状态、下一步该怎么走”。
# 颜色按语义：红 = 需要人处理，黄 = 等待重跑，蓝 = 完成，灰 = 还没开始。
states = {"pending": (1.2, 2.2, "待处理\n数据到了", FAINT), "blocked": (4.6, 3.6, "已阻断\n门禁不通过", RED),
          "done": (4.6, 0.8, "已完成", MAIN), "stale": (8.0, 2.2, "待重跑", AMBER), "failed": (1.2, 0.2, "出错", RED)}
fig, ax = plt.subplots(figsize=(10, 5.6))
for k, (x, y, lab, c) in states.items():
    ax.add_patch(FancyBboxPatch((x - 0.75, y - 0.38), 1.5, 0.76, boxstyle="round,pad=0.02,rounding_size=0.14", fc=c, ec="none"))
    ax.text(x, y, lab, ha="center", va="center", color="white", fontsize=10.5, fontweight="bold", linespacing=1.3)
# 每条边单独定弯曲方向和标签位置：两条反向的边分开弯，标签不再叠在一起
arrows = [("pending", "blocked", "门禁不通过", -0.2, (2.55, 3.35)), ("pending", "done", "门禁通过、处理完成", 0.2, (2.75, 1.05)),
          ("blocked", "stale", "收到补传", -0.2, (6.6, 3.4)), ("done", "stale", "收到迟到数据\n或前一天被重跑", 0.35, (7.15, 0.75)),
          ("stale", "done", "run --todo 重跑", 0.0, (5.95, 1.72)), ("pending", "failed", "代码出错", 0.0, (1.6, 1.2)),
          ("failed", "done", "修复后重跑", 0.0, (2.9, 0.38))]
for a, b, lab, rad, (lx, ly) in arrows:
    ax.add_patch(FancyArrowPatch(states[a][:2], states[b][:2], connectionstyle=f"arc3,rad={rad}", arrowstyle="-|>",
                                 mutation_scale=13, color=MUTED, lw=1.2, shrinkA=30, shrinkB=30))
    ax.text(lx, ly, lab, ha="left" if a == "pending" and b == "failed" else "center", va="center", fontsize=9.5,
            color=INK, bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))
ax.set_xlim(0, 9.2); ax.set_ylim(-0.4, 4.2); ax.axis("off")
viz.finish(fig, "每个分区只有 5 种状态，run --todo 只处理待处理、待重跑和出错的",
           "分区 = 一天的数据；状态记在 partition_registry；门禁不通过的分区下游一行都不写；红 = 需要人处理，黄 = 等待重跑，蓝 = 完成",
           SOURCE, "D12_01_分区生命周期.png")

In [ ]:
from signal_mysql import database as mysql
cell_keys = set(mysql.read_frame("SELECT lac_cell FROM dim_cell", wh)["lac_cell"])
profiles = {dt: gate.profile_partition(pipeline.extract_day(wh, dt), cell_keys)
            for dt in ["2018-10-01", "2018-10-02", "2018-10-03"]}
prof = pd.DataFrame(profiles).T.drop(columns="hourly")
prof["有数据的小时数"] = [sum(h > 0 for h in p["hourly"]) for p in profiles.values()]
prof

In [ ]:
# ── 出图 D12_02：三条折线（同一刻度）+ 条形图 ─────────────────────────
# 为什么三天必须画在同一个刻度上：分开缩放会让 10-02 看起来像正常的一天，而这正是本图要反驳的错觉。
base_hour = profiles[DAY]["rows"] / 24
col = {"2018-10-01": FAINT, "2018-10-02": RED, "2018-10-03": MAIN}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5.6), width_ratios=[1.6, 1])
a1.axhline(0.2 * base_hour / 1e4, color=MUTED, ls="--", lw=1)
a1.text(0.2, 0.2 * base_hour / 1e4 + 0.06, f"门禁线：10-03 小时均值的 20%（{0.2 * base_hour:,.0f} 条）", fontsize=9, color=MUTED, va="bottom")
ends = []
for dt, p in profiles.items():
    h = np.array(p["hourly"])
    a1.plot(range(24), h / 1e4, color=col[dt], lw=2.4 if dt != "2018-10-01" else 1.6, marker="o", ms=3)
    ends.append([h[-1] / 1e4, dt[5:], col[dt]])
ends.sort()
for i in range(1, len(ends)):
    ends[i][0] = max(ends[i][0], ends[i - 1][0] + 0.25)
for y, lab, c in ends:
    a1.text(23.6, y, lab, color=c, fontsize=10, fontweight="bold", va="center")
a1.set_xlim(-0.5, 25.5); a1.set_ylim(0, base_hour / 1e4 * 1.7)
a1.set_xticks(range(0, 24, 3), [f"{x} 点" for x in range(0, 24, 3)])
a1.set_title("每小时记录数（万条，三天同一刻度）", loc="left", fontsize=10.5, color=INK, fontweight="bold")
for i, (dt_, p) in enumerate(profiles.items()):
    rpu = p["rows"] / max(p["users"], 1); vh = int((np.array(p["hourly"]) > 0).sum())
    a2.barh(i, rpu, color=col[dt_], height=0.56)
    a2.text(rpu + 2, i, f"{rpu:.1f} 条/人 · 有数据 {vh} 小时", va="center", fontsize=9.5, color=INK,
            fontweight="bold" if dt_ == "2018-10-02" else "normal")
a2.set_yticks(range(len(profiles)), [d[5:] for d in profiles], fontsize=10); a2.invert_yaxis()
a2.set_xlim(0, 175); a2.set_xticks([]); a2.grid(False)
a2.set_title("人均记录数", loc="left", fontsize=10.5, color=INK, fontweight="bold")
p2 = profiles["2018-10-02"]
viz.finish(fig, f"10-02 每个小时都有数据，人均却只有 {p2['rows'] / p2['users']:.1f} 条：“有数据”不等于“数据够”",
           "三个真实分区的画像；红色 = 门禁不通过的 10-02；只检查“每小时有没有数据”，10-02 会被当成完整的一天",
           SOURCE, "D12_02_三天画像.png")

In [ ]:
# ════ 基准为什么用中位数，不用平均数 ════
# 造 21 天模拟数据，第 6 天采集故障（30%）、第 10 天节日高峰（160%），看两种基准各自被影响多久。
# 基准只能用“当天之前”的数据：数据泄漏是监控和建模都要避开的坑。
rng = np.random.default_rng(7)
days = pd.date_range("2018-11-01", periods=21)
vol = rng.normal(1.0, 0.04, len(days))
vol[5], vol[9] = 0.30, 1.60
series = pd.Series(vol * 518_443, index=days)
win = 7
past = series.shift(1).rolling(win, min_periods=3)
mean_base, med_base = past.mean(), past.median()
flag_mean = series < 0.5 * mean_base
flag_med = series < 0.5 * med_base
print(f"第 6 天之后一周，平均数基准最低被拉到 {mean_base.iloc[6:13].min() / 518_443:.0%}，"
      f"中位数基准最低 {med_base.iloc[6:13].min() / 518_443:.0%}")

In [ ]:
# ── 出图 D12_03：柱状图 + 两条基准线 ────────────────────────────────
# 柱 = 每天实际多少（离散观测），线 = 基准（连续参考）；两类语义不同的东西用不同图形编码。
fig, ax = plt.subplots(figsize=(10, 5.6), dpi=100)
ax.bar(series.index, series / 1e4, color=[RED if f else GRAY for f in flag_med], width=0.7,
       hatch="///", edgecolor="white", lw=0)                                  # 斜线 + 水印 = 模拟数据
ax.plot(mean_base.index, mean_base / 1e4, color=AMBER, lw=2)
ax.plot(med_base.index, med_base / 1e4, color=MAIN, lw=2.6)
x_end = series.index[-1] + pd.Timedelta(hours=14)
ys = sorted([(mean_base.iloc[-1] / 1e4, "基准 = 前 7 天平均数", AMBER), (med_base.iloc[-1] / 1e4, "基准 = 前 7 天中位数", MAIN)])
ys[1] = (max(ys[1][0], ys[0][0] + 5), ys[1][1], ys[1][2])
for y, lab, c in ys:
    ax.text(x_end, y, lab, color=c, fontsize=10, fontweight="bold", va="center")
low_mean, low_med = mean_base.iloc[6:13].min() / 518_443, med_base.iloc[6:13].min() / 518_443
ax.annotate("第 6 天采集故障（30%）", xy=(series.index[5], series.iloc[5] / 1e4), xytext=(series.index[1], 22),
            fontsize=9.5, color=RED, arrowprops=dict(arrowstyle="-", color=RED, lw=0.8))
ax.annotate(f"之后一周平均数基准最低被拉到 {low_mean:.0%}", xy=(series.index[8], mean_base.iloc[8] / 1e4),
            xytext=(series.index[11], 76), fontsize=9.5, color=AMBER, arrowprops=dict(arrowstyle="-", color=AMBER, lw=0.8))
ax.set_xlim(series.index[0] - pd.Timedelta(days=1), series.index[-1] + pd.Timedelta(days=5)); ax.set_ylim(0, 92)
ax.set_xticks(series.index[::4]); ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%m-%d"))
ax.set_ylabel("记录数（万条）")
viz.watermark(fig)
viz.finish(fig, f"一个故障日会把平均数基准拉低一周（最低 {low_mean:.0%}），中位数基准几乎不动（最低 {low_med:.0%}）",
           "模拟的 21 天数据量（斜线 = 模拟数据）：第 6 天采集故障，第 10 天节日高峰（160%）；基准只用当天之前的 7 天；\n"
           "红色 = 低于中位数基准的一半，被门禁拦下；平均数对异常值敏感，下一次故障可能因此“看起来没那么低”而漏报",
           SOURCE, "D12_03_基准选择.png")

In [ ]:
# 每张表这一天的内容哈希：行数 + 按全部列排序后的哈希和。
TABLES = ["dwd_signal", "dws_stay", "dws_night", "dws_trip", "dws_user_day", "dws_user_hour"]
def fingerprint(dt):
    out = {}
    for t in TABLES:
        df = mysql.read_frame(f"SELECT * FROM {t} WHERE dt = ?", wh, params=(dt,))
        cols = [c for c in df.columns if c not in ("cross_day", "duration_in_day_min")]    # D12 新加的两列不参与比较
        out[t] = (len(df), int(pd.util.hash_pandas_object(df[cols].sort_values(cols), index=False).sum()))
    return out
fp_before = fingerprint(DAY)
cli("run", "--todo")

In [ ]:
from signal_mysql import database as mysql
fp_after = fingerprint(DAY)
print("10-03 的 6 张表完全没变：", fp_before == fp_after)
mysql.read_frame("SELECT dt, rule_id, rule_name, value, threshold, result FROM dq_gate_result WHERE dt = '2018-10-02'", wh)

In [ ]:
from signal_mysql import database as mysql
# ── 出图 D12_04：门禁结果热力图 ─────────────────────────────────────
# 行 = 分区，列 = 规则；红 = 不通过、黄 = 警告、浅蓝 = 通过；格子里写实际值。
res = mysql.read_frame("SELECT dt, rule_id, rule_name, value, result FROM dq_gate_result", wh)
dates = sorted(res["dt"].unique()); rules = sorted(res["rule_id"].unique())
color = {"fail": RED, "warn": AMBER, "pass": MAIN_LIGHT}
fig, ax = plt.subplots(figsize=(10, 1.6 + 0.62 * len(dates)))
for i, d in enumerate(dates):
    for j, r in enumerate(rules):
        row = res[(res["dt"] == d) & (res["rule_id"] == r)].iloc[0]
        ax.add_patch(plt.Rectangle((j, i), 0.96, 0.9, color=color[row["result"]]))
        v = row["value"]
        txt = f"{v:.0f}" if r == "G03" else f"{v:.1%}" if v < 10 else f"{v:,.0f}"
        ax.text(j + 0.48, i + 0.45, txt, ha="center", va="center", fontsize=9,
                color="white" if row["result"] == "fail" else INK)
names = res.drop_duplicates("rule_id").set_index("rule_id")["rule_name"]
ax.set_xticks([j + 0.48 for j in range(len(rules))], [f"{r}\n{names[r]}" for r in rules], fontsize=8.5)
ax.set_yticks([i + 0.45 for i in range(len(dates))], dates, fontsize=10)
ax.set_xlim(0, len(rules)); ax.set_ylim(len(dates), 0); ax.grid(False)
for sp in ax.spines.values():
    sp.set_visible(False)
blocked = res.loc[res["result"] == "fail", "dt"].nunique()
viz.finish(fig, f"{len(dates)} 个分区里 {blocked} 个被门禁拦下，拦下它们的都是“量”的规则 G01–G03",
           "红 = 不通过，黄 = 警告，浅蓝 = 通过；格子里是实际值（比率、有效小时数）；任何一条不通过，整个分区不通过、下游一行都不写",
           SOURCE, "D12_04_门禁结果.png")

In [ ]:
# 契约测试在文件 tests/contract/test_d12_gate.py 里；这里挑三条最关键的在 notebook 里复核一遍。
assert gate.overall(gate.evaluate(
    gate.profile_partition(pipeline.extract_day(wh, "2018-10-02"), cell_keys),
    gate.baseline(pd.DataFrame(columns=["rows", "rows_per_user"]), settings), settings)) == "fail"
assert gate.overall(gate.evaluate(
    gate.profile_partition(pipeline.extract_day(wh, "2018-10-03"), cell_keys),
    gate.baseline(pd.DataFrame(columns=["rows", "rows_per_user"]), settings), settings)) in ("pass", "warn")
assert settings["gate"]["rules"][0]["id"] == "G01" and len(settings["gate"]["rules"]) == 7
print("✅ 门禁契约自检通过：10-02 不通过、10-03 通过、7 条规则都在配置里")

In [ ]:
for d in ["2018-10-04", "2018-10-05"]:
    f = next((x for x in [config.LANDING_DIR / f"ods_signal_{d}.parquet", config.LANDING_DIR / f"ods_signal_{d}.db"] if x.exists()), None)
    assert f is not None, f"找不到 {d} 的分区文件：把讲师发的分区文件放进 data/landing/"
    cli("ingest", str(f))
cli("run", "--todo")

In [ ]:
from signal_mysql import database as mysql
D4 = "2018-10-04"
cell = mysql.read_frame("SELECT lac_cell, site_id, lng, lat, district FROM dim_cell", wh)
districts = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_district_boundary", wh), "district")
aois = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_aoi_v2", wh), "name")
hot = mysql.read_frame("SELECT * FROM dim_hotspot", wh).iloc[0]
since = (pd.Timestamp(D4) - pd.Timedelta(hours=settings["stay"]["lookback_hours"])).strftime("%Y-%m-%d %H:%M:%S")
sig_day = mysql.read_frame("SELECT imsi, ts, lac_cell FROM dwd_signal WHERE dt = ?", wh, params=(D4,), parse_dates=["ts"]).merge(cell, on="lac_cell")
lookback = mysql.read_frame("SELECT imsi, ts, lac_cell FROM dwd_signal WHERE dt = ? AND ts >= ?", wh,
                            params=(DAY, since), parse_dates=["ts"]).merge(cell, on="lac_cell")
with_lb = dws.build_stay(pd.concat([lookback, sig_day], ignore_index=True), settings, districts, aois, hot, D4)
without = dws.build_stay(sig_day, settings, districts, aois, hot, D4)
nights = {k: dws.build_night(v, D4, settings) for k, v in [("没有回看", without), ("有回看", with_lb)]}
compare = pd.DataFrame({k: [len(v), int(v["cross_day"].sum()), len(nights[k]), nights[k]["night_min"].mean()]
                        for k, v in [("没有回看", without), ("有回看", with_lb)]},
                       index=["停留数", "跨午夜的停留", "识别出过夜地的人", "过夜重叠时长均值（分钟）"]).round(1)
compare

In [ ]:
x = with_lb[with_lb["cross_day"]].copy()
x["before_mid"] = (pd.Timestamp(D4) - x["start_time"]).dt.total_seconds() / 3600
n_stays = with_lb.groupby("imsi").size()
x = x[x["imsi"].map(n_stays) <= 3]
uid = x.sort_values("before_mid").iloc[len(x) // 2]["imsi"]
t0 = pd.Timestamp(D4) - pd.Timedelta(hours=6)
hrs = lambda t: (t - t0).total_seconds() / 3600
fig, ax = plt.subplots(figsize=(10, 4.4))
for yi, (lab, st) in enumerate([("有回看", with_lb), ("没有回看", without)]):
    for _, s in st[st["imsi"] == uid].iterrows():
        a, b = max(hrs(s["start_time"]), 0), hrs(s["end_time"])
        ax.barh(yi, b - a, left=a, height=0.46, color=MAIN if s["cross_day"] else MAIN_MID, hatch="///",
                edgecolor="white", lw=0)
        if b - a >= 2.5:
            ax.text(a + (b - a) / 2, yi, f"{s['start_time']:%H:%M}–{s['end_time']:%H:%M}", ha="center", va="center",
                    color="white", fontsize=9.5, fontweight="bold")
lb_part = lookback[lookback["imsi"] == uid]
ax.scatter([hrs(t) for t in lb_part["ts"]], [-0.48] * len(lb_part), marker="|", color=MUTED, s=70)
ax.text(0.1, -0.62, "回看窗口里的信令（前一天 18 点起）", fontsize=9, color=MUTED, va="bottom")
ax.axvline(6, color=INK, lw=1, ls="--"); ax.text(6.15, 1.42, "午夜", fontsize=9.5, color=INK)
ax.set_yticks([0, 1], ["有回看", "没有回看"]); ax.set_ylim(-0.8, 1.7); ax.set_xlim(0, 12)
ax.set_xticks(range(0, 13, 2), [f"{(18 + h) % 24} 点" for h in range(0, 13, 2)])
ax.set_ylabel("")
viz.finish(fig, "没有回看，跨午夜的停留被切成两段；有回看才拼得回来",
           f"一个人（尾号 {privacy.mask_id(uid)}）从 10-04 前半夜到当天的停留；斜线 = 从前一天开始的停留；竖线 = 午夜",
           SOURCE, "D12_05_回看窗口.png")

In [ ]:
f06 = next((x for x in [config.LANDING_DIR / "ods_signal_2018-10-06.parquet", config.LANDING_DIR / "ods_signal_2018-10-06.db"] if x.exists()), None)
assert f06 is not None, "找不到 10-06 的分区文件"
cli("ingest", str(f06))
cli("run", "--todo")

In [ ]:
late = next((x for x in [config.LANDING_DIR / "ods_signal_2018-10-06_补传.parquet", config.LANDING_DIR / "ods_signal_2018-10-06_补传.db"] if x.exists()), None)
assert late is not None, "找不到 10-06 补传文件"
cli("ingest", str(late))                                       # 运营商补传了 0–13 点
r06 = ingest.reg_get(wh, "2018-10-06")
print({k: r06[k] for k in ["gate_status", "pipeline_status", "note"]})
cli("run", "--todo")

In [ ]:
from signal_mysql import database as mysql
cli("ingest", str(late))                                       # 同一个文件又被送来一次：按指纹跳过
prof06 = mysql.read_frame("SELECT dt, rows, rows_per_user, baseline_rows FROM dq_partition_profile WHERE dt = '2018-10-06'", wh)
print(f"10-06 现在的数据量是基准的 {prof06['rows'].iloc[0] / prof06['baseline_rows'].iloc[0]:.0%}")
mysql.read_frame("SELECT source_file AS file, `rows`, dates, is_synthetic, loaded_at FROM ods_load_log", wh)

In [ ]:
cli("run", "--start", "2018-10-03", "--end", "2018-10-05")
reg = ingest.registry(wh)
print("回填之后变成“待重跑”的分区：", reg.loc[reg["pipeline_status"] == "stale", ["dt", "note"]].to_dict("records"))
cli("run", "--todo")

In [ ]:
from signal_mysql import database as mysql
log = mysql.read_frame("SELECT dt, run_id, step, elapsed_sec AS seconds FROM etl_log", wh)
last = log.groupby("dt")["run_id"].max()
log = log[log["run_id"].isin(last)]
grp = {"extract_day": "抽取", "quality_gate": "门禁", "clean_signal": "清洗", "write_dwd": "清洗"}
log["组"] = log["step"].map(lambda s: grp.get(s, "汇总层"))
order = ["抽取", "门禁", "清洗", "汇总层"]
piv = log.pivot_table(index="dt", columns="组", values="seconds", aggfunc="sum").reindex(columns=order).fillna(0)
piv = piv[piv.sum(axis=1) > 5].sort_index(ascending=False)
syn = dict(mysql.read_frame("SELECT dt, is_synthetic FROM partition_registry", wh).values)
shades = ["#174E73", "#8E99A6", MAIN, MAIN_MID]
fig, ax = plt.subplots(figsize=(10, 5.6))
for i, (d, r) in enumerate(piv.iterrows()):
    left = 0
    for k, c in zip(order, shades):
        ax.barh(i, r[k], left=left, color=c, height=0.58, hatch="///" if syn.get(d) else None, edgecolor="white", lw=0.6)
        if i == len(piv) - 1:
            ax.text(left + r[k] / 2, i - 0.42, k, ha="center", va="bottom", fontsize=9, color=INK)
        left += r[k]
    ax.text(left + 0.15, i, f"{left:.1f} 秒" + ("（模拟）" if syn.get(d) else ""), va="center", fontsize=10,
            fontweight="normal" if syn.get(d) else "bold")
ax.set_yticks(range(len(piv)), piv.index, fontsize=10)
ax.set_xlim(0, piv.sum(axis=1).max() * 1.25); ax.set_ylim(len(piv) - 0.4, -0.6)
ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.set_xlabel("秒")
mean_s = piv.sum(axis=1).mean()
viz.watermark(fig, "含模拟分区 · 仅用于流水线演示")
viz.finish(fig, f"每个分区约 {mean_s:.0f} 秒，一年的数据约 {mean_s * 365 / 60:.0f} 分钟就能回填完",
           "各分区最近一次运行的耗时（etl_log，与机器有关）；汇总层 = 五张 dws_ 表的计算和写入；斜线 = 模拟分区；\n"
           "耗时只和当天的数据量有关，不随历史天数增长，因为抽取按时间范围走索引、写入按分区先删后写",
           SOURCE, "D12_06_回填耗时.png")

In [ ]:
# 用 EXPLAIN 对比有索引和没索引的执行计划：出现 SCAN 就是慢的信号，Index lookup 才是走了索引。
wh.execute("DROP TABLE IF EXISTS cell_noidx")
wh.execute("CREATE TEMPORARY TABLE cell_noidx AS SELECT * FROM dim_cell")      # 一份没有索引的维表
sql = "SELECT s.imsi, c.lng FROM dwd_signal s JOIN {t} c ON s.lac_cell = c.lac_cell WHERE s.dt = ?"
for t in ["cell_noidx", "dim_cell"]:
    plan = wh.execute("EXPLAIN FORMAT=TREE " + sql.format(t=t), (DAY,)).fetchone()[0]
    print(f"{t:<11}", plan.splitlines()[0])

In [ ]:
import time
cut = wh.execute("SELECT imsi FROM (SELECT DISTINCT imsi FROM dwd_signal WHERE dt = ? ORDER BY imsi LIMIT 20) t ORDER BY imsi DESC LIMIT 1", (DAY,)).fetchone()[0]
part = sql + " AND s.imsi <= ?"                                  # 只取前 20 个用户，否则要等很久
n_try = wh.execute("SELECT COUNT(*) FROM dwd_signal WHERE dt = ? AND imsi <= ?", (DAY, cut)).fetchone()[0]
t = time.perf_counter()
wh.execute("SELECT COUNT(*) FROM (" + part.format(t="cell_noidx") + ") t", (DAY, cut)).fetchone()
slow = time.perf_counter() - t
n_day = wh.execute("SELECT COUNT(*) FROM dwd_signal WHERE dt = ?", (DAY,)).fetchone()[0]
t = time.perf_counter()
wh.execute("SELECT COUNT(*) FROM (" + sql.format(t="dim_cell") + ") t", (DAY,)).fetchone()
fast = time.perf_counter() - t
print(f"没有索引：{n_try:,} 行用了 {slow:.1f} 秒，推算全天 {n_day:,} 行要 {slow / n_try * n_day / 60:.0f} 分钟")
print(f"有索引：全天 {n_day:,} 行用了 {fast:.2f} 秒，快了约 {slow / n_try * n_day / fast:,.0f} 倍")

In [ ]:
from signal_mysql import database as mysql
from signal_insight.geo import haversine
# MySQL 版用自连接取相邻停留（pandas 里是 groupby().shift(-1)），距离在 pandas 里算。
pairs = mysql.read_frame(
    "SELECT a.imsi, a.lat, a.lng, b.lat AS next_lat, b.lng AS next_lng FROM dws_stay a "
    "JOIN dws_stay b ON a.dt = b.dt AND a.imsi = b.imsi AND b.stay_seq = a.stay_seq + 1 WHERE a.dt = ?", wh, params=(DAY,))
d = haversine(pairs["lat"], pairs["lng"], pairs["next_lat"], pairs["next_lng"])
by_sql = (int((d.round() >= settings["trip"]["min_dist_m"]).sum()), int(pairs.loc[d.round() >= settings["trip"]["min_dist_m"], "imsi"].nunique()))
by_pd = wh.execute("SELECT COUNT(*), COUNT(DISTINCT imsi) FROM dws_trip WHERE dt = ?", (DAY,)).fetchone()
print(f"SQL：{by_sql[0]:,} 次出行 / {by_sql[1]:,} 人；pandas：{by_pd[0]:,} 次 / {by_pd[1]:,} 人")

In [ ]:
CONTRACTS = {
    "dws_stay": {"pk": ["dt", "imsi", "stay_seq"], "not_null": ["start_time", "end_time", "lng", "lat"],
                 "checks": {"时长 ≥ 最短停留": f"duration_min >= {settings['stay']['min_minutes']}",
                            "开始 ≤ 结束": "start_time <= end_time", "经度在东北": "lng BETWEEN 118 AND 128"}},
    "dws_user_hour": {"pk": ["dt", "imsi", "hour"], "not_null": ["lng", "lat"],
                      "checks": {"小时在 0–23": "hour BETWEEN 0 AND 23", "来源只有两种": "src IN ('停留','信令')"}},
}
rows = []
for t, c in CONTRACTS.items():
    pk = ",".join(c["pk"])
    dup = wh.execute(f"SELECT COUNT(*) FROM (SELECT {pk} FROM {t} GROUP BY {pk} HAVING COUNT(*) > 1) t").fetchone()[0]
    rows.append({"表": t, "契约": f"主键（{pk}）唯一", "违反行数": dup})
    for col in c["not_null"]:
        rows.append({"表": t, "契约": f"{col} 非空", "违反行数": wh.execute(f"SELECT COUNT(*) FROM {t} WHERE {col} IS NULL").fetchone()[0]})
    for name, cond in c["checks"].items():
        rows.append({"表": t, "契约": name, "违反行数": wh.execute(f"SELECT COUNT(*) FROM {t} WHERE NOT ({cond})").fetchone()[0]})
contract = pd.DataFrame(rows)
contract["结果"] = np.where(contract["违反行数"] == 0, "✅", "❌")
contract

In [ ]:
# 今日对账
reg = ingest.registry(wh).set_index("dt")
assert fp_before == fingerprint(DAY), "10-03 的结果被改动了"
assert (reg.loc[["2018-10-01", "2018-10-02"], "pipeline_status"] == "blocked").all()
assert (reg.loc[["2018-10-03", "2018-10-04", "2018-10-05", "2018-10-06"], "pipeline_status"] == "done").all()
assert reg.loc[["2018-10-04", "2018-10-05", "2018-10-06"], "is_synthetic"].all()
assert compare.loc["跨午夜的停留", "没有回看"] == 0 and compare.loc["跨午夜的停留", "有回看"] > 1_500
assert by_sql[0] == by_pd[0] == 11_055 and (contract["违反行数"] == 0).all()
print("✅ 对账通过：10-01、10-02 被拦下；10-03 自 D11 以来一个字节没变；三个模拟分区处理完成；"
      f"跨午夜停留 0 → {int(compare.loc['跨午夜的停留', '有回看']):,}；SQL 与 pandas 出行数一致；数据契约全部满足")
wh.close(); con.close()